In [1]:
import os
import numpy as np 
import torch
import pandas as pd
from utils.read_oridata import read_stress_graph, remove_z_and_global_normalize # 确保这里引入的是你文件名对应的方法

In [2]:
#输入.csv数据集目录
root = r'D:\composite_0602\initial_dataset'
#输入保存图数据集目录
save_dir = r'D:\composite_0602\graph_dataset'


# 确保保存目录存在
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    
graphs = []
processed_cases = []

# 获取所有案例名称
all_node_files = [f for f in os.listdir(root) if f.endswith(' nodes.csv')]

print(f"开始扫描，共发现 {len(all_node_files)} 个潜在案例文件")

开始扫描，共发现 149 个潜在案例文件


In [3]:
for file_name in all_node_files:
    # 直接提取案例名
    case = file_name.replace(' nodes.csv', '')
    
    node_path = os.path.join(root, file_name)
    edge_path = os.path.join(root, f"{case} edges.csv")
    
    if not os.path.exists(edge_path):
        print(f"[跳过] 找不到对应的边文件: {case}")
        continue
    
    try:
        node_df = pd.read_csv(node_path)
        edge_df = pd.read_csv(edge_path)

        """
        为什么统一要求符合 11371 节点和 33714 边
        """
        
        # 严格校验：统一要求符合 11371 节点和 33714 边
        if len(node_df) != 11371 or len(edge_df) != 33714:
            print(f"[不合格] {case} | 节点: {len(node_df)}, 边: {len(edge_df)}")
            continue
            
        graph = read_stress_graph(node_path, edge_path)
        
        if graph is not None:
            graphs.append(graph)
            processed_cases.append(case)
            print(f"[成功] {case}")
            
    except Exception as e:
        print(f"[异常] {case}: {e}")

# 保存
dataset_date = "20260602" #更新测试日期
save_path = os.path.join(save_dir, f"{dataset_date}_{len(graphs)}cases.pt")
# torch.save(graphs, save_path)

# print(f"\n--- 处理完成 ---")
# print(f"总计提取成功案例: {len(graphs)} 个")
# print(f"文件已保存: {save_path}")

[成功] 1.1
[成功] 1.2
[成功] 10.1
[成功] 11.1
[成功] 11.2
[成功] 12.1
[成功] 12.2
[成功] 13.1
[成功] 13.2
[成功] 14.1
[成功] 14.2
[成功] 15.1
[成功] 15.2
[成功] 16.1
[成功] 16.2
[成功] 17.1
[成功] 17.2
[成功] 2.1
[成功] 2.2
[成功] 3.1
[成功] 3.2
[成功] 4.1
[成功] 4.2
[不合格] 5.1 | 节点: 0, 边: 0
[成功] 5.2
[成功] 6.1
[成功] 6.2
[成功] 7.1
[成功] 7.2
[成功] 8.1
[成功] 8.2
[成功] 9.1
[成功] 9.2
[成功] jia1-s3-1
[成功] jia1-s3-2
[不合格] jia10-s2-1 | 节点: 11380, 边: 33743
[不合格] jia10-s2-2 | 节点: 11380, 边: 33743
[不合格] jia11-s1-1 | 节点: 11380, 边: 33743
[不合格] jia11-s1-2 | 节点: 11380, 边: 33743
[不合格] jia12-s3-1 | 节点: 11380, 边: 33743
[不合格] jia12-s3-2 | 节点: 11380, 边: 33743
[不合格] jia13-s1-1 | 节点: 11380, 边: 33743
[不合格] jia13-s1-2 | 节点: 11380, 边: 33743
[不合格] jia14-s3-1 | 节点: 11380, 边: 33743
[不合格] jia14-s3-2 | 节点: 11380, 边: 33743
[不合格] jia15-s1-1 | 节点: 11380, 边: 33743
[不合格] jia15-s1-2 | 节点: 11380, 边: 33743
[不合格] jia16-s5-1 | 节点: 11380, 边: 33743
[不合格] jia16-s5-2 | 节点: 11380, 边: 33743
[不合格] jia17-s1-1 | 节点: 11380, 边: 33743
[不合格] jia17-s1-2 | 节点: 11380, 边: 33743
[不合格] jia18-s3-1 | 

In [5]:
processed_graphs = remove_z_and_global_normalize(graphs)
# save_path = os.path.join(save_dir, f"{dataset_date}_{len(graphs)}_pro.pt")
# torch.save(processed_graphs, save_path)

In [10]:
processed_graphs[0]

Data(x=[11371, 12], edge_index=[2, 67428], y_node=[11371, 1], y_edge=[67428, 1])

In [13]:
from torch_geometric.utils import degree, clustering_coeff, to_undirected

def add_structural_features(data):
    # 假设 edge_index 是双向有向边，先转为无向
    edge_index_undirected = to_undirected(data.edge_index)
    deg = degree(edge_index_undirected[0], num_nodes=data.num_nodes).unsqueeze(1)  # (N,1)
    
    # 聚类系数（可能需要安装 torch-cluster）
    try:
        clust = clustering_coeff(data.edge_index, num_nodes=data.num_nodes).unsqueeze(1)
    except:
        clust = torch.zeros((data.num_nodes, 1))
    
    # 归一化度（除以平均度）
    deg_norm = deg / deg.mean()
    
    # 拼接新特征
    data.x = torch.cat([data.x, deg_norm, clust], dim=1)
    return data

ImportError: cannot import name 'clustering_coeff' from 'torch_geometric.utils' (D:\ProgramData\miniconda3\envs\torch24_py312\Lib\site-packages\torch_geometric\utils\__init__.py)

In [12]:
add_structural_features(processed_graphs[0])

NameError: name 'to_undirected' is not defined

In [9]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ========== 配置 ==========
DATA_PATH = r"D:\composite_0602\graph_dataset\20260602_104_pro.pt"
OUTPUT_DIR = './dataset_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========== 加载数据集 ==========
print("正在加载数据集...")
dataset = torch.load(DATA_PATH, weights_only=False)
print(f"数据集包含 {len(dataset)} 个图\n")

# ========== 辅助函数 ==========
def tensor_stats(data, name):
    """输出一维数据的统计信息，支持 numpy 数组或 torch 张量"""
    if torch.is_tensor(data):
        arr = data.numpy().ravel()
    else:
        arr = np.array(data).ravel()
    stats = {
        'mean': np.mean(arr),
        'std': np.std(arr),
        'min': np.min(arr),
        'max': np.max(arr),
        'q25': np.percentile(arr, 25),
        'q50': np.percentile(arr, 50),
        'q75': np.percentile(arr, 75),
        'zero_ratio': np.mean(arr == 0)
    }
    print(f"\n--- {name} 统计 ---")
    for k, v in stats.items():
        print(f"{k:8s}: {v:.6f}")
    return stats

def plot_histogram(data, title, xlabel, save_name, bins=50):
    """绘制直方图并保存"""
    plt.figure(figsize=(8, 5))
    plt.hist(data, bins=bins, edgecolor='black', alpha=0.7)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('频数')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.close()
    print(f"  直方图已保存至 {save_name}")

# ========== 1. 全局基础信息 ==========
print("=" * 60)
print("1. 全局基础信息")
print("=" * 60)
num_graphs = len(dataset)
total_nodes = sum(data.x.size(0) for data in dataset)
total_edges = sum(data.edge_index.size(1) for data in dataset)
print(f"图数量: {num_graphs}")
print(f"总节点数: {total_nodes}")
print(f"总边数（有向边，每条无向边对应两条有向边）: {total_edges}")
print(f"平均每图节点数: {total_nodes / num_graphs:.1f}")
print(f"平均每图边数: {total_edges / num_graphs:.1f}")

# ========== 2. 节点标签 y_node 统计 ==========
print("\n" + "=" * 60)
print("2. 节点标签 (y_node) 统计")
print("=" * 60)
all_y_node = torch.cat([data.y_node for data in dataset], dim=0)  # 保持张量
y_stats = tensor_stats(all_y_node, "y_node (奇异点概率)")
plot_histogram(all_y_node.numpy().ravel(), "节点标签分布 (y_node)", "概率", "y_node_hist.png", bins=100)

# 每个图内部的 y_node 均值分布
per_graph_mean_y = [data.y_node.mean().item() for data in dataset]
per_graph_max_y = [data.y_node.max().item() for data in dataset]
print(f"\n每个图内 y_node 均值的分布: min={np.min(per_graph_mean_y):.6f}, max={np.max(per_graph_mean_y):.6f}, mean={np.mean(per_graph_mean_y):.6f}")
print(f"每个图内 y_node 最大值的分布: min={np.min(per_graph_max_y):.6f}, max={np.max(per_graph_max_y):.6f}, mean={np.mean(per_graph_max_y):.6f}")

non_zero_ratio = np.mean((all_y_node.numpy() > 0.01))
print(f"y_node > 0.01 的比例: {non_zero_ratio:.4f}")

# ========== 3. 边标签 y_edge 统计 ==========
print("\n" + "=" * 60)
print("3. 边标签 (y_edge) 统计")
print("=" * 60)
all_y_edge = torch.cat([data.y_edge for data in dataset], dim=0)
edge_stats = tensor_stats(all_y_edge, "y_edge (应力线概率)")
plot_histogram(all_y_edge.numpy().ravel(), "边标签分布 (y_edge)", "概率", "y_edge_hist.png", bins=100)

per_graph_mean_edge = [data.y_edge.mean().item() for data in dataset]
print(f"\n每个图内 y_edge 均值的分布: min={np.min(per_graph_mean_edge):.6f}, max={np.max(per_graph_mean_edge):.6f}, mean={np.mean(per_graph_mean_edge):.6f}")

# ========== 4. 节点特征 x 统计 ==========
print("\n" + "=" * 60)
print("4. 节点特征 (x) 各列统计")
print("=" * 60)
# 特征列名（12维，顺序根据实际预处理结果）
feature_names = [
    'x', 'y',
    'm1_vx', 'm1_vy', 'm1_abs',
    'm1_type_+t', 'm1_type_-c',
    'm2_vx', 'm2_vy', 'm2_abs',
    'm2_type_+t', 'm2_type_-c'
]
all_x = torch.cat([data.x for data in dataset], dim=0).numpy()
print(f"节点特征矩阵形状: {all_x.shape}")

col_stats = []
for i, name in enumerate(feature_names):
    col_data = all_x[:, i]
    stats = {
        'mean': np.mean(col_data),
        'std': np.std(col_data),
        'min': np.min(col_data),
        'max': np.max(col_data),
        'q25': np.percentile(col_data, 25),
        'q75': np.percentile(col_data, 75)
    }
    col_stats.append(stats)
    print(f"\n[{i:2d}] {name:15s}: mean={stats['mean']:.4f}, std={stats['std']:.4f}, min={stats['min']:.4f}, max={stats['max']:.4f}")

# 关键特征直方图
key_features = ['m1_abs', 'm2_abs', 'm1_vx', 'm2_vx']
for name in key_features:
    idx = feature_names.index(name)
    plot_histogram(all_x[:, idx], f"特征分布: {name}", name, f"x_{name}_hist.png", bins=50)

# ========== 5. 图结构统计 ==========
print("\n" + "=" * 60)
print("5. 图结构统计")
print("=" * 60)
avg_degrees = []
for data in dataset:
    deg = torch.bincount(data.edge_index[0], minlength=data.num_nodes).float().mean().item()
    avg_degrees.append(deg)
print(f"平均节点度（有向边）: mean={np.mean(avg_degrees):.2f}, std={np.std(avg_degrees):.2f}")

# 节点特征与 y_node 的 Pearson 相关系数
print("\n节点特征与 y_node 的 Pearson 相关系数（全局）:")
all_y_np = all_y_node.numpy().ravel()
for i, name in enumerate(feature_names):
    corr = np.corrcoef(all_x[:, i], all_y_np)[0, 1]
    print(f"  {name:15s}: {corr:.4f}")

# ========== 6. 保存统计表 ==========
df_y = pd.DataFrame({
    'y_node': all_y_node.numpy().ravel(),
    'y_edge': all_y_edge.numpy().ravel()
})
df_y.describe().to_csv(os.path.join(OUTPUT_DIR, 'label_statistics.csv'))
print(f"\n详细标签统计已保存至 {OUTPUT_DIR}/label_statistics.csv")

df_feat = pd.DataFrame(all_x, columns=feature_names)
df_feat.describe().to_csv(os.path.join(OUTPUT_DIR, 'feature_statistics.csv'))
print(f"特征统计已保存至 {OUTPUT_DIR}/feature_statistics.csv")

print("\n分析完成！")

正在加载数据集...
数据集包含 104 个图

1. 全局基础信息
图数量: 104
总节点数: 1182584
总边数（有向边，每条无向边对应两条有向边）: 7012512
平均每图节点数: 11371.0
平均每图边数: 67428.0

2. 节点标签 (y_node) 统计

--- y_node (奇异点概率) 统计 ---
mean    : 0.000298
std     : 0.013749
min     : 0.000000
max     : 1.000000
q25     : 0.000000
q50     : 0.000000
q75     : 0.000000
zero_ratio: 0.995471
  直方图已保存至 y_node_hist.png

每个图内 y_node 均值的分布: min=0.000116, max=0.000629, mean=0.000298
每个图内 y_node 最大值的分布: min=1.000000, max=1.000000, mean=1.000000
y_node > 0.01 的比例: 0.0011

3. 边标签 (y_edge) 统计


C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 39057 (\N{CJK UNIFIED IDEOGRAPH-9891}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 33410 (\N{CJK UNIFIED IDEOGRAPH-8282}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 28857 (\N{CJK UNIFIED IDEOGRAPH-70B9}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 26631 (\N{CJK UNIFIED IDEOGRAPH-6


--- y_edge (应力线概率) 统计 ---
mean    : 0.007936
std     : 0.063703
min     : 0.000000
max     : 1.000000
q25     : 0.000000
q50     : 0.000000
q75     : 0.000000
zero_ratio: 0.895975
  直方图已保存至 y_edge_hist.png

每个图内 y_edge 均值的分布: min=0.002615, max=0.022027, mean=0.007936

4. 节点特征 (x) 各列统计
节点特征矩阵形状: (1182584, 12)

[ 0] x              : mean=0.5004, std=0.2912, min=0.0000, max=1.0000

[ 1] y              : mean=0.5007, std=0.2914, min=0.0000, max=1.0000

[ 2] m1_vx          : mean=0.5751, std=0.1815, min=0.0000, max=1.0000


C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 36793 (\N{CJK UNIFIED IDEOGRAPH-8FB9}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)



[ 3] m1_vy          : mean=0.5039, std=0.1622, min=0.0000, max=1.0000

[ 4] m1_abs         : mean=0.1279, std=0.1240, min=0.0000, max=1.0000

[ 5] m1_type_+t     : mean=0.8566, std=0.3505, min=0.0000, max=1.0000

[ 6] m1_type_-c     : mean=0.1434, std=0.3505, min=0.0000, max=1.0000

[ 7] m2_vx          : mean=0.4987, std=0.3208, min=0.0000, max=1.0000

[ 8] m2_vy          : mean=0.4554, std=0.3480, min=0.0000, max=1.0000

[ 9] m2_abs         : mean=0.1314, std=0.1451, min=0.0000, max=1.0000

[10] m2_type_+t     : mean=0.3197, std=0.4663, min=0.0000, max=1.0000

[11] m2_type_-c     : mean=0.6803, std=0.4663, min=0.0000, max=1.0000


C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 29305 (\N{CJK UNIFIED IDEOGRAPH-7279}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
C:\Users\Karos\AppData\Local\Temp\ipykernel_26672\2127480001.py:47: UserWarning: Glyph 24449 (\N{CJK UNIFIED IDEOGRAPH-5F81}) missing from font(s) DejaVu Sans.
  plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)


  直方图已保存至 x_m1_abs_hist.png
  直方图已保存至 x_m2_abs_hist.png
  直方图已保存至 x_m1_vx_hist.png
  直方图已保存至 x_m2_vx_hist.png

5. 图结构统计
平均节点度（有向边）: mean=5.93, std=0.00

节点特征与 y_node 的 Pearson 相关系数（全局）:
  x              : 0.0022
  y              : 0.0018
  m1_vx          : -0.0028
  m1_vy          : -0.0003
  m1_abs         : 0.0168
  m1_type_+t     : -0.0077
  m1_type_-c     : 0.0077
  m2_vx          : 0.0003
  m2_vy          : 0.0089
  m2_abs         : 0.0131
  m2_type_+t     : 0.0165
  m2_type_-c     : -0.0165


ValueError: All arrays must be of the same length

In [7]:
!pip install seaborn

In [38]:
from torch_geometric.data import Data

def remove_z_and_global_normalize(graph_list):
    """
    对图列表中每个图执行：
    1. 删除节点特征中的 z 列（原索引 2）
    2. 对 m1_vx, m1_vy, m1_abs, m2_vx, m2_vy, m2_abs（新索引 [2,3,4,7,8,9]）进行全局 Min-Max 归一化（跨所有图）
    3. 对每个图的 y_node（is_singularity）进行**单图内部** Min-Max 归一化到 [0,1]
    返回新的图列表，其中节点特征维度为 12（无 z），且 y_node 已按图归一化。
    """
    if not graph_list:
        return []

    # 1. 去除 z 列，创建新图列表
    graphs_no_z = []
    for data in graph_list:
        x_no_z = torch.cat([data.x[:, :2], data.x[:, 3:]], dim=1)
        new_data = Data(
            x=x_no_z,
            edge_index=data.edge_index,
            y_node=data.y_node.clone(),      # 克隆保留原始值
            y_edge=data.y_edge
        )
        graphs_no_z.append(new_data)

    # 2. 全局归一化节点特征 (x) 中的选定列（跨所有图）
    norm_cols = [2, 3, 4, 7, 8, 9]  # m1_vx, m1_vy, m1_abs, m2_vx, m2_vy, m2_abs
    col_min = [float('inf')] * len(norm_cols)
    col_max = [float('-inf')] * len(norm_cols)

    for data in graphs_no_z:
        x = data.x
        for i, col in enumerate(norm_cols):
            col_data = x[:, col]
            min_val = col_data.min().item()
            max_val = col_data.max().item()
            if min_val < col_min[i]:
                col_min[i] = min_val
            if max_val > col_max[i]:
                col_max[i] = max_val

    for data in graphs_no_z:
        x = data.x
        for i, col in enumerate(norm_cols):
            min_val = col_min[i]
            max_val = col_max[i]
            if max_val - min_val > 1e-8:
                x[:, col] = (x[:, col] - min_val) / (max_val - min_val)
            else:
                x[:, col] = torch.zeros_like(x[:, col])

    # 3. 单图内部归一化 y_node（每个图独立）
    for data in graphs_no_z:
        y = data.y_node
        if y.numel() == 0:
            continue
        y_min = y.min()
        y_max = y.max()
        if y_max - y_min > 1e-8:
            data.y_node = (y - y_min) / (y_max - y_min)
        else:
            data.y_node = torch.zeros_like(y)

    return graphs_no_z